# Phase 4C.1 — Learning Curve
Launcher thực thi ma trận 15 runs Stage 1 (frozen) trên Colab T4 kèm lưu trữ bền vững Google Drive.

## Thiết lập

In [ ]:
from pathlib import Path

EXECUTE = False
DOWNLOAD_FINAL_ARCHIVE = False

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab/phase_4c1")
DRIVE_INPUT_DIR = DRIVE_ROOT / "inputs"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "runs" / "execution_79bb115"

LOCAL_TRANSFER_DIR = Path("/content/forensics-transfer")
LOCAL_OUTPUT_PATH = Path("/content/phase_4c1_outputs")

EXPECTED_ARTIFACTS = {
    "code_archive": {
        "filename": "phase_4c1_code_79bb115.tar.gz",
        "bytes": 758350,
        "sha256": "5e775a7708d1555bf22f56dceb5358e26e07dd224c4b6186f575aff4d2b590d5"
    },
    "reusable_bundle": {
        "filename": "phase_4c1_binary_n250_reusable.tar",
        "bytes": 724633600,
        "sha256": "d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27"
    },
    "operator_script": {
        "filename": "phase_4c1_t4_execute_all_stage1.sh",
        "bytes": 21662,
        "sha256": "1a7570d757ccfc1b471c636f64d0f001c3b6fcc9306b9894f94186f909f1f3c4"
    }
}

print(f"[READY] Cấu hình hoàn tất: EXECUTE={EXECUTE}, DOWNLOAD={DOWNLOAD_FINAL_ARCHIVE}")

## Kiểm tra môi trường và dữ liệu

In [ ]:
import sys
import shutil
import hashlib
import torch
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("[SKIP] google.colab không khả dụng, bỏ qua mount Drive")

assert sys.platform.startswith("linux"), f"Yêu cầu Linux, nhận được: {sys.platform}"

gpu_model = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
if EXECUTE:
    assert torch.cuda.is_available(), "CUDA không khả dụng khi EXECUTE=True"
    assert "T4" in gpu_model.upper(), f"Yêu cầu NVIDIA T4, nhận được: {gpu_model}"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_INPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stat_content = shutil.disk_usage("/content") if Path("/content").exists() else shutil.disk_usage(".")
free_gb = stat_content.free / (1024 ** 3)
if EXECUTE:
    assert free_gb >= 5.0, f"Dung lượng đĩa /content không đủ: {free_gb:.2f} GB, cần >= 5.0 GB"

def compute_sha256_streaming(file_path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest().lower()

missing_artifacts = []
for key, spec in EXPECTED_ARTIFACTS.items():
    fp = DRIVE_INPUT_DIR / spec["filename"]
    if not fp.exists():
        missing_artifacts.append(spec["filename"])
        continue
    act_bytes = fp.stat().st_size
    assert act_bytes == spec["bytes"], f"Sai số byte: {spec['filename']}"
    act_sha = compute_sha256_streaming(fp)
    assert act_sha == spec["sha256"].lower(), f"Sai SHA-256: {spec['filename']}"
    print(f"[PASS] {spec['filename']}")

if missing_artifacts:
    raise FileNotFoundError(f"Thiếu artifact trong Drive inputs: {missing_artifacts}")

print("[PASS] Kiểm tra môi trường và dữ liệu thành công")

## Chuẩn bị phiên huấn luyện

In [ ]:
import os
import shutil
from pathlib import Path

LOCAL_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)

for key, spec in EXPECTED_ARTIFACTS.items():
    src = DRIVE_INPUT_DIR / spec["filename"]
    dst = LOCAL_TRANSFER_DIR / spec["filename"]
    need_copy = True
    if dst.exists():
        if dst.stat().st_size == spec["bytes"] and compute_sha256_streaming(dst) == spec["sha256"].lower():
            print(f"[PASS] Tái sử dụng tệp cục bộ: {spec['filename']}")
            need_copy = False
        else:
            raise ValueError(f"Tệp cục bộ {dst} sai byte hoặc SHA-256")
    if need_copy:
        part_dst = LOCAL_TRANSFER_DIR / f"{spec['filename']}.part"
        if part_dst.exists():
            part_dst.unlink()
        shutil.copy2(src, part_dst)
        assert part_dst.stat().st_size == spec["bytes"], f"Lỗi copy {part_dst.name}"
        assert compute_sha256_streaming(part_dst) == spec["sha256"].lower(), f"Lỗi SHA-256 {part_dst.name}"
        part_dst.rename(dst)
        print(f"[COPY] {spec['filename']}")

operator_local = LOCAL_TRANSFER_DIR / "phase_4c1_t4_execute_all_stage1.sh"
operator_local.chmod(0o755)

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_OUTPUT_PATH.exists() or LOCAL_OUTPUT_PATH.is_symlink():
    if LOCAL_OUTPUT_PATH.is_symlink():
        target_resolved = os.path.realpath(LOCAL_OUTPUT_PATH)
        expected_resolved = os.path.realpath(DRIVE_OUTPUT_DIR)
        assert target_resolved == expected_resolved, f"Symlink trỏ sai: {target_resolved}"
        print(f"[PASS] Symlink đầu ra: {LOCAL_OUTPUT_PATH} -> {DRIVE_OUTPUT_DIR}")
    else:
        raise RuntimeError(f"{LOCAL_OUTPUT_PATH} là thư mục vật lý, không phải symlink")
else:
    LOCAL_OUTPUT_PATH.symlink_to(DRIVE_OUTPUT_DIR, target_is_directory=True)
    print(f"[PASS] Tạo symlink đầu ra: {LOCAL_OUTPUT_PATH} -> {DRIVE_OUTPUT_DIR}")

print("[READY] Chuẩn bị phiên huấn luyện hoàn tất")

## Huấn luyện

In [ ]:
import subprocess

if not EXECUTE:
    print("[SKIP] EXECUTE=False: Bỏ qua thực thi huấn luyện")
    print("[READY] Đổi EXECUTE = True tại mục Thiết lập để chạy 15 runs Stage 1")
else:
    operator_cmd = ["bash", str(LOCAL_TRANSFER_DIR / "phase_4c1_t4_execute_all_stage1.sh")]
    print(f"[RUN] {' '.join(operator_cmd)}")
    try:
        subprocess.run(operator_cmd, check=True)
        print("[PASS] Huấn luyện hoàn thành thành công")
    except subprocess.CalledProcessError as e:
        print(f"[FAIL] Operator thoát với mã lỗi: {e.returncode}")
        raise

## Kết quả

In [ ]:
import json
from pathlib import Path

REQUIRED_ARCHIVES = [
    "n50_results.tar.gz",
    "n100_results.tar.gz",
    "n250_results.tar.gz",
    "phase_4c1_t4_execution_logs.tar.gz",
    "phase_4c1_all_15_runs_results.tar.gz"
]

if not EXECUTE:
    print("[SKIP] EXECUTE=False: Bỏ qua kiểm tra kết quả")
else:
    drive_download_dir = DRIVE_OUTPUT_DIR / "download"
    assert drive_download_dir.exists(), f"Thư mục download thiếu: {drive_download_dir}"

    for name in REQUIRED_ARCHIVES:
        archive_path = drive_download_dir / name
        sidecar_path = drive_download_dir / f"{name}.sha256"
        assert archive_path.exists(), f"Thiếu archive: {archive_path}"
        assert sidecar_path.exists(), f"Thiếu sidecar: {sidecar_path}"
        bytes_count = archive_path.stat().st_size
        assert bytes_count > 0, f"Archive rỗng: {archive_path}"
        expected_hash = sidecar_path.read_text(encoding="utf-8").strip().split()[0].lower()
        actual_hash = compute_sha256_streaming(archive_path)
        assert actual_hash == expected_hash, f"Sai SHA-256 archive: {name}"
        print(f"[PASS] {name} ({bytes_count:,} bytes)")

    completed_runs = 0
    for size in (50, 100, 250):
        for seed in (42, 1337, 2025, 3407, 9001):
            run_dir = DRIVE_OUTPUT_DIR / f"n{size}_seed_{seed}"
            receipt_path = run_dir / "run_receipt.json"
            assert receipt_path.exists(), f"Thiếu receipt: {run_dir.name}"
            with open(receipt_path, "r", encoding="utf-8") as f:
                receipt = json.load(f)
            assert receipt.get("status") == "completed", f"Trạng thái chưa hoàn thành: {run_dir.name}"
            assert receipt.get("sample_size") == size, f"Sai sample_size: {run_dir.name}"
            assert receipt.get("seed") == seed, f"Sai seed: {run_dir.name}"
            assert receipt.get("stage") == "frozen", f"Sai stage: {run_dir.name}"
            assert receipt.get("locked_test_access", -1) == 0, f"Locked-test access vi phạm: {run_dir.name}"
            assert receipt.get("stage2_invocations", -1) == 0, f"Stage 2 vi phạm: {run_dir.name}"
            assert receipt.get("validation_source_count") == 91, f"Sai số validation sources: {run_dir.name}"
            completed_runs += 1

    assert completed_runs == 15, f"Kỳ vọng 15 runs, nhận được: {completed_runs}"
    print(f"[PASS] Xác thực hoàn tất {completed_runs}/15 runs trong Drive: {DRIVE_OUTPUT_DIR}")

    if DOWNLOAD_FINAL_ARCHIVE:
        final_tar = drive_download_dir / "phase_4c1_all_15_runs_results.tar.gz"
        print(f"[COPY] Tải {final_tar.name} về máy...")
        try:
            from google.colab import files
            files.download(str(final_tar))
        except ImportError:
            print("[SKIP] google.colab.files không khả dụng")